# Generation: Generating a Response

In [2]:
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [3]:
embeddings = OllamaEmbeddings(model = "nomic-embed-text")

vectorstore = Chroma(
    persist_directory='./intro-to-ds-lectures',
    embedding_function=embeddings
)

/var/folders/ll/jc6zxxdd0d11gx258tqz2x8m0000gn/T/ipykernel_19295/3270883534.py:1: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(model = "nomic-embed-text")
/var/folders/ll/jc6zxxdd0d11gx258tqz2x8m0000gn/T/ipykernel_19295/3270883534.py:3: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


In [4]:
len(vectorstore.get()['documents'])

22

In [5]:
retriever = vectorstore.as_retriever(
    search_type = 'mmr',
    search_kwargs = {
        'k' : 3,
        'lambda_mult' : 0.7
    }
)

In [6]:
TEMPLATE = '''
Answer the following question:
{question}

To answer the question, use only the following context:
{context}

At the end of the response, specify the name of the lecture this context is taken from in the format:
Resources: *Lecture Title*
where *Lecture Title* should be substituted with the title of all resource lectures.
'''

prompt_template = PromptTemplate.from_template(TEMPLATE)

In [7]:
chat = ChatOpenAI(
    model_name='llama3.2:3b',
    openai_api_key='ollama', 
    openai_api_base='http://localhost:11434/v1',
    temperature = 0, 
    max_tokens = 250,
    model_kwargs = {
        'seed':365
    }
)

/opt/anaconda3/envs/ai-ml/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3519: UserWarning: Parameters {'seed'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  if await self.run_code(code, result, async_=asy):


In [8]:
question = "What software do data scientists use?"

In [9]:
chain = (
    {
        'context': retriever, 
        'question': RunnablePassthrough()
    } | prompt_template | chat | StrOutputParser()
)

In [11]:
result  = chain.invoke(question)
print(result)

Based on the provided context, data scientists use the following software:

1. R
2. Python

These two programming languages are mentioned as the most popular tools across all columns in the infographic and are highlighted for their ability to manipulate data and be integrated within multiple data science software platforms.

Resources: Programming Languages & Software Employed in Data Science - All the Tools You Need
